In this notebook we will cluster a set molecules using the [Taylor-Butina](https://pubs.acs.org/doi/pdf/10.1021/ci9803381) clustering method.  

In [1]:
%pip install pandas rdkit seaborn tqdm mols2grid


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Import the necessary Python libraries

In [2]:
import pandas as pd
from rdkit import Chem
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import Descriptors
from tqdm.auto import tqdm
import mols2grid

In [5]:
# від арабського «taqaddum» (прогрес)
tqdm.pandas()

Define a function to do the clustering.  While the RDKit has a function to do the clustering, we still need to cacluate fingerprints and assign cluster ids. This function performs the following steps. 
1. Calculate fingerprints for each molecule in mol_list
2. Calculate the similarity of every molecule to every other molecule (all pairs)
3. Create a distance matrix containing 1-similarity for each pairwise similarity value
4. Assign a cluster id to each molecule

In [7]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina


def butina_cluster(mol_list, cutoff=0.35):
    """Cluster molecules using the Taylor-Butina algorithm.

    Groups structurally similar molecules together based on Morgan
    fingerprint similarity. Unlike k-means, the number of clusters is not
    fixed in advance; it emerges from the data given a similarity cutoff.
    Molecules with no close neighbours form their own single-member
    clusters (singletons).

    Args:
        mol_list (list[rdkit.Chem.Mol]): Molecules to cluster, as RDKit Mol
            objects.
        cutoff (float, optional): Tanimoto distance threshold (1 - Tanimoto
            similarity). Two molecules are placed in the same cluster only if
            their distance is below this value. Lower cutoffs yield more,
            tighter clusters; higher cutoffs yield fewer, looser ones.
            Defaults to 0.35.

    Returns:
        list[int]: A cluster ID for each input molecule, in the same order as
        ``mol_list``. IDs start at 1, and the most populated cluster is
        labelled 1.

    Example:
        >>> from rdkit import Chem
        >>> mols = [Chem.MolFromSmiles(s) for s in ["CCO", "CCN", "c1ccccc1"]]
        >>> butina_cluster(mols, cutoff=0.4)
        [1, 1, 2]
    """
    # Create a Morgan fingerprint generator (radius 3, 2048 bits)
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)

    # Generate a fingerprint for every molecule
    fp_list = [generator.GetFingerprint(m) for m in mol_list]

    # Build a flattened lower-triangular distance matrix.
    # For each molecule i, compute its Tanimoto similarity to all earlier
    # molecules (0..i-1) at once, then convert similarity to distance (1 - sim).
    dists = []
    nfps = len(fp_list)
    for i in range(1, nfps):
        sims = DataStructs.BulkTanimotoSimilarity(fp_list[i], fp_list[:i])
        dists.extend([1 - x for x in sims])

    # Run Butina clustering on the precomputed distances
    mol_clusters = Butina.ClusterData(dists, nfps, cutoff, isDistData=True)

    # Map each molecule index to its cluster ID (1-based)
    cluster_id_list = [0] * nfps
    for idx, cluster in enumerate(mol_clusters, 1):
        for member in cluster:
            cluster_id_list[member] = idx

    return cluster_id_list

Read a csv file with the input data. This data contains a set of ERK2 inhibitors and decoy molecules from the DUD-E database.

In [10]:
import sys
sys.path.append("..")

from read_csv import read_csv
df = read_csv("https://raw.githubusercontent.com/PatWalters/practical_cheminformatics_tutorials/main/data/dude_erk2_mk01.csv")

Let's look at the first few lines of the dataframe.  Note that there are three columns, SMILES, molecule name, and an indicator variable "is_active" where 1 represents an active compound and 0 represents a decoy. 

In [11]:
df.head()

,Unnamed: 0,SMILES,ID,is_active
0,0,Cn1ccnc1Sc2ccc(cc2Cl)Nc3c4cc(c(cc4ncc3C#N)OCCC...,168691,1
1,1,C[C@@]12[C@@H]([C@@H](CC(O1)n3c4ccccc4c5c3c6n2...,86358,1
2,2,Cc1cnc(nc1c2cc([nH]c2)C(=O)N[C@H](CO)c3cccc(c3...,575087,1
3,3,Cc1cnc(nc1c2cc([nH]c2)C(=O)N[C@H](CO)c3cccc(c3...,575065,1
4,4,Cc1cnc(nc1c2cc([nH]c2)C(=O)N[C@H](CO)c3cccc(c3...,575047,1


In [12]:
mols2grid.display(df)

Add an RDKit molecule column to the dataframe. 

In [13]:
df['mol'] = df.SMILES.progress_apply(Chem.MolFromSmiles)

  0%|          | 0/4629 [00:00<?, ?it/s]

Cluster the molecules in the dataframe and assign the cluster id to a new column. 

In [14]:
%time df['Cluster'] = butina_cluster(df.mol.values)

CPU times: user 2.34 s, sys: 363 ms, total: 2.7 s
Wall time: 2.93 s


View the dataframe with the new **Cluster** column

In [15]:
mols2grid.display(df,subset=["img","ID","Cluster"])

Let's look at how we could select the molecule from each cluster with the lowest LogP.  First we'll calculate the LogP for each molecule and put these values into a new column called "logP".

In [16]:
df["logP"] = df.mol.progress_apply(Descriptors.MolLogP)

  0%|          | 0/4629 [00:00<?, ?it/s]

View the dataframe with the LogP added. Note how we use the mols2grid **transform** function to set the number of decimal places for LogP.   Try removing this and see what happens, it's not pretty. 

In [17]:
mols2grid.display(df,subset=["img","ID","Cluster","logP"],transform={"logP": lambda x: f"{x:.2f}"})

Sort the dataframe, first by **Cluster** then by **logP**.

In [18]:
df.sort_values(["Cluster","logP"],inplace=True)

In [19]:
mols2grid.display(df,subset=["img","ID","Cluster","logP"],transform={"logP": lambda x: f"{x:.2f}"})

Now let's create a new dataframe containing only the molecule from each cluster with the lowest LogP.  Since we have already sorted the dataframe by cluster id and LogP, we can simply use the [drop_duplicates](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop_duplicates.html) method to drop all but the first row in each cluster. 

In [20]:
df_unique = df.drop_duplicates("Cluster")

In [21]:
mols2grid.display(df_unique,subset=["img","ID","Cluster","logP"],transform={"logP": lambda x: f"{x:.2f}"})

# Taylor-Butina Clustering Notebook — ELI5 Summary

Here's what you did in this notebook, in plain language.

## The big idea

You have a set of 4629 molecules — ERK2 inhibitors (compounds that can block a particular protein target) and "decoys" (similar-looking but inactive molecules) from the DUD-E database. The goal was: **group similar molecules together using the Taylor-Butina method, then pick one "best" representative from each group**.

Imagine a huge box of candy. First you sort it into piles by similarity (chocolates with chocolates, jellies with jellies), then from each pile you take one candy based on some criterion. That's what you were doing.

## Step by step

**1. Wrote the clustering function `butina_cluster`**
This is the function with the docstring we formatted. It does four things: computes a fingerprint for each molecule, compares every molecule to every other (Tanimoto similarity), converts similarity to distance (1 − similarity), and assigns each molecule a cluster number.

**2. Loaded the data**
Read the CSV through your own `read_csv` (with `sys.path.append("..")`, because the notebook lives in the `clustering/` folder while the function is one level up). The table has three main columns: `SMILES` (structure), `ID` (name), and `is_active` — an indicator where **1 = active compound**, **0 = decoy**.

**3. Turned SMILES into RDKit molecules**
```python
df['mol'] = df.SMILES.progress_apply(Chem.MolFromSmiles)
```
The SMILES text strings are turned into full RDKit molecule objects, which you can then use to compute fingerprints and properties. (`progress_apply` is the tqdm version with the progress bar we discussed.)

**4. Clustered**
```python
%time df['Cluster'] = butina_cluster(df.mol.values)
```
Ran your function on all the molecules — each got a cluster number in a new `Cluster` column. `%time` is a Jupyter "magic command" that measured how long it took (~3 seconds).

**5. Computed LogP for each molecule**
```python
df["logP"] = df.mol.progress_apply(Descriptors.MolLogP)
```
**LogP** is a measure of lipophilicity — how "greasy" / poorly water-soluble a molecule is. It's an important property in drug discovery: too high a LogP is often a bad sign. Here it's computed for each molecule and stored in the `logP` column.

A small display detail:
```python
transform={"logP": lambda x: f"{x:.2f}"}
```
This just rounds LogP to 2 decimal places when shown in the grid — so you don't get ugly long numbers. (This, by the way, is a real **f-string** — the same thing you earlier confused with a docstring.)

**6. Sorted: first by Cluster, then by LogP**
```python
df.sort_values(["Cluster","logP"], inplace=True)
```
Now the table is ordered so that within each cluster the molecules go from lowest LogP to highest. So **the first row of each cluster is the molecule with the lowest LogP** in that cluster.

**7. Picked one representative from each cluster**
```python
df_unique = df.drop_duplicates("Cluster")
```
`drop_duplicates("Cluster")` keeps **only the first row for each Cluster value**. And since you just sorted by LogP, the first row = the molecule with the lowest LogP. So you got a table with one molecule per cluster, each with the best (lowest) LogP.

This is a small but clever trick: instead of writing complicated "find the minimum in each group" logic, you just **sort + drop duplicates**, and it works.

## Why do all this

In drug discovery you often have thousands of similar molecules. Testing them all is expensive and pointless, because many are nearly identical. So:

1. **Group** by similarity (Taylor-Butina) → you get rough "families" of molecules.
2. **Take one representative from each family** by some criterion (here, lowest LogP, since that's often a desirable property).

The result: instead of 4629 molecules, you have a short list of unique representatives that covers the full chemical diversity of the set, but without thousands of near-duplicates. This saves time, money, and reagents.

## What was new here (compared to the k-means notebook)

- A different clustering algorithm — **Taylor-Butina** (no need to specify the number of clusters, just a threshold).
- Computing a molecular property — **LogP** via `Descriptors.MolLogP`.
- A neat **sort + drop_duplicates** trick for picking the "best of each group".
- `%time` for measuring speed, and `transform` in mols2grid for formatting numbers.